# 07.02 · Graph API: state, nodes, edges, reducers

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 07.01. |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Define** graph state with `TypedDict` or Pydantic, and know what each choice validates.
2. **Control merges** with reducers: overwrite (default), `operator.add`, `add_messages`, and custom ones.
3. **Use** `MessagesState` for chat-shaped graphs and runtime `context` for per-run settings.
4. **Wire** nodes with normal and conditional edges, `START` and `END`; compile, invoke and stream.
5. **Visualize** a graph inline (PNG, Mermaid, ASCII) and read a step-by-step state trace.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Graph API overview](https://docs.langchain.com/oss/python/langgraph/graph-api) | State, schemas, reducers, nodes, edges, `START`/`END`, super-steps |
| [Use the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) | How-tos: reducers, `MessagesState`, Pydantic state, runtime context, visualization |
| [INVALID_CONCURRENT_GRAPH_UPDATE](https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE) | The error you'll trigger on purpose in Part 2 |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "grandalf>=0.8,<1"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain-core",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · The smallest useful graph

Three building blocks: **state** (a schema), **nodes** (functions `state -> partial update`), **edges** (who runs next). `compile()` checks the wiring and returns a runnable with `invoke`, `stream` and `batch`, like any LangChain runnable.

In [ ]:
# 🧪 Offline: state, two nodes, three edges.
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class Order(TypedDict):
    item: str
    quantity: int
    unit_price: float
    total: float
    message: str


def price(state: Order) -> dict:
    return {"total": round(state["quantity"] * state["unit_price"], 2)}


def confirm(state: Order) -> dict:
    return {"message": f"{state['quantity']} × {state['item']} = US${state['total']}"}


builder = StateGraph(Order)
builder.add_node(price)
builder.add_node(confirm)
builder.add_edge(START, "price")
builder.add_edge("price", "confirm")
builder.add_edge("confirm", END)
order_graph = builder.compile()

result = order_graph.invoke({"item": "Nimbus Mini", "quantity": 2, "unit_price": 89.0})
print(result)

**You should now see** `{'item': 'Nimbus Mini', 'quantity': 2, 'unit_price': 89.0, 'total': 178.0, 'message': '2 × Nimbus Mini = US$178.0'}`. The input didn't include `total` or `message`: nodes added them. `invoke` returns the **whole final state**, not just the last node's output.

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? `show_graph` prints a ⚠️ and shows the Mermaid text instead.

In [ ]:
# 🌐 Draw it. show_graph tries a PNG (mermaid.ink), then falls back to Mermaid text. ASCII needs grandalf.
from mastery.graphs import show_graph

print("rendered as:", show_graph(order_graph))
try:
    order_graph.get_graph().print_ascii()
except ImportError:
    print("ASCII drawing needs grandalf: %pip install grandalf")

**You should now see** a small diagram `__start__ → price → confirm → __end__` (as an image, or as Mermaid text after a ⚠️ if you're offline), and the same graph in ASCII art.

`show_graph` lives in `mastery/graphs/nodes.py`: it's `display(Image(graph.get_graph().draw_mermaid_png()))` from the docs, plus the offline fallback. The Mermaid text is also what you'd paste into a Markdown cell or a README.

---
## Part 2 · Reducers: how updates are merged

After every node, LangGraph merges the node's update into the state **key by key**. Without a reducer the new value **overwrites** the old one. With a reducer, `new = reducer(old, update)`. You declare it with `Annotated[type, reducer]`.

In [ ]:
# 🧪 Offline: the same two nodes, with and without a reducer on `log`.
import operator
from typing import Annotated


class NoReducer(TypedDict, total=False):
    log: list[str]


class WithReducer(TypedDict, total=False):
    log: Annotated[list[str], operator.add]     # concatenate lists


def step_a(state) -> dict:
    return {"log": ["a ran"]}


def step_b(state) -> dict:
    return {"log": ["b ran"]}


def build(schema):
    b = StateGraph(schema)
    b.add_node("a", step_a)
    b.add_node("b", step_b)
    b.add_edge(START, "a")
    b.add_edge("a", "b")
    b.add_edge("b", END)
    return b.compile()


print("overwrite :", build(NoReducer).invoke({"log": ["start"]}))
print("operator.add:", build(WithReducer).invoke({"log": ["start"]}))

**You should now see** `overwrite : {'log': ['b ran']}` and `operator.add: {'log': ['start', 'a ran', 'b ran']}`. Same nodes, different state schema, different result: the merge rule belongs to the **state**, not to the nodes.

Now the error every LangGraph user meets once: two nodes that run **in the same step** (in parallel) write the same key, and there's no reducer to combine them.

In [ ]:
# 🧪 Offline: parallel writers. START → a and START → b run in the same super-step.
from langgraph.errors import InvalidUpdateError


def build_parallel(schema):
    b = StateGraph(schema)
    b.add_node("a", step_a)
    b.add_node("b", step_b)
    b.add_edge(START, "a")
    b.add_edge(START, "b")
    b.add_edge(["a", "b"], END)       # wait for both
    return b.compile()


try:
    build_parallel(NoReducer).invoke({"log": []})
except InvalidUpdateError as exc:
    print("InvalidUpdateError:", str(exc).splitlines()[0])
print("with reducer:", build_parallel(WithReducer).invoke({"log": []}))

**You should now see** `InvalidUpdateError: At key 'log': Can receive only one value per step. Use an Annotated key to handle multiple values.` and then `with reducer: {'log': ['a ran', 'b ran']}`.

### What just happened?

LangGraph runs in **super-steps**: all nodes that are ready run together, then all their updates are applied. Two values for one key in the same super-step need a rule to combine them. Sequential nodes never hit this, which is why the error shows up the day you add parallelism (07.03).

**Custom reducers** are plain functions `(old, new) -> merged`. `mastery.graphs` has three you'll reuse:

In [ ]:
# 🧪 Offline: reusable reducers from mastery.graphs.state.
from mastery.graphs import append_unique, keep_last, merge_dicts

print(merge_dicts({"model": "a", "k": 4}, {"k": 6}))         # settings: later keys win
print(append_unique(["pro.md"], ["pro.md", "mini.md"]))     # sources without duplicates
print(keep_last(3)([1, 2, 3], [4, 5]))                     # a bounded log


class Research(TypedDict, total=False):
    sources: Annotated[list[str], append_unique]
    recent: Annotated[list[int], keep_last(3)]


def found(state) -> dict:
    return {"sources": ["mini.md", "pro.md"], "recent": [4, 5]}


rb = StateGraph(Research)
rb.add_node(found)
rb.add_edge(START, "found")
print(rb.compile().invoke({"sources": ["pro.md"], "recent": [1, 2, 3]}))

**You should now see** `{'model': 'a', 'k': 6}`, `['pro.md', 'mini.md']`, `[3, 4, 5]`, and the graph result `{'sources': ['pro.md', 'mini.md'], 'recent': [3, 4, 5]}`.

> 💡 **The input goes through the reducer too.** `invoke({"sources": ["pro.md"]})` is applied like an update, so with `operator.add` re-running a graph on its own output would duplicate items. (`Overwrite(value)` from `langgraph.types` bypasses a reducer for one update when you really need to replace.)

---
## Part 3 · `MessagesState` and `add_messages`

Chat-shaped graphs keep a list of messages. `add_messages` is the reducer for that: it **appends** new messages, **replaces** a message whose `id` already exists, and accepts tuples like `("user", "hi")`. `MessagesState` is a ready-made state with `messages: Annotated[list, add_messages]`; subclass it to add keys.

In [ ]:
# 🧪 Offline: add_messages appends by default and replaces by id.
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages

history = [HumanMessage("What's the Mini's price?", id="m1"), AIMessage("US$89.", id="m2")]
print([m.content for m in add_messages(history, [AIMessage("US$89 (VAT incl.)", id="m2")])])   # replace m2
print([m.content for m in add_messages(history, ("user", "And the Pro?"))])                  # append


class ChatState(MessagesState):
    turns: int


def answer(state: ChatState) -> dict:
    return {"messages": [AIMessage(f"You said: {state['messages'][-1].content}")], "turns": state.get("turns", 0) + 1}


cb = StateGraph(ChatState)
cb.add_node(answer)
cb.add_edge(START, "answer")
chat = cb.compile()
out = chat.invoke({"messages": [("user", "hello")]})
print([f"{m.type}: {m.content}" for m in out["messages"]], "| turns:", out["turns"])

**You should now see** `["What's the Mini's price?", 'US$89 (VAT incl.)']`, then the history plus `'And the Pro?'`, then `['human: hello', 'ai: You said: hello'] | turns: 1`. The tuple `("user", "hello")` became a `HumanMessage`.

---
## Part 4 · Pydantic state and runtime context

`TypedDict` is a type **hint**: nothing is checked at run time. A Pydantic model as state **validates the graph input** (not every node's output). Nodes receive the model instance (`state.n`), and `invoke` still returns a dict.

**Runtime context** is for per-run settings that aren't state (user id, model choice, feature flags): declare `context_schema`, pass `context=` to `invoke`, read `runtime.context` in a node.

In [ ]:
# 🧪 Offline: Pydantic validates the input; context carries per-run settings.
from dataclasses import dataclass

from pydantic import BaseModel, Field, ValidationError
from langgraph.runtime import Runtime


class Quote(BaseModel):
    quantity: int = Field(ge=1)
    unit_price: float
    total: float = 0.0


@dataclass
class Pricing:
    currency: str = "USD"
    discount: float = 0.0


def quote(state: Quote, runtime: Runtime[Pricing]) -> dict:
    gross = state.quantity * state.unit_price
    return {"total": round(gross * (1 - runtime.context.discount), 2)}


qb = StateGraph(Quote, context_schema=Pricing)
qb.add_node(quote)
qb.add_edge(START, "quote")
quote_graph = qb.compile()

print(quote_graph.invoke({"quantity": 3, "unit_price": 89.0}, context=Pricing(discount=0.1)))
try:
    quote_graph.invoke({"quantity": 0, "unit_price": 89.0})
except ValidationError as exc:
    print("ValidationError:", exc.errors()[0]["msg"])

**You should now see** `{'quantity': 3, 'unit_price': 89.0, 'total': 240.3}` and `ValidationError: Input should be greater than or equal to 1`.

| State schema | Checks at run time | Use when |
|---|---|---|
| `TypedDict` | nothing (type hints only) | most graphs; fastest; what the docs use |
| Pydantic `BaseModel` | the **graph input**; defaults filled | input comes from outside (an API, Studio, a user) |
| `dataclass` | nothing; gives defaults | you want defaults without Pydantic |

---
## Part 5 · Conditional edges, and reading a run step by step

A **conditional edge** calls a function on the state and goes where it says. Pass the possible targets (a list, or a dict `{returned_value: node}`) so the graph can be drawn and checked before it runs.

In [ ]:
# 🧪 Offline: route large orders to a review node; stream the run node by node.
import operator

from mastery.graphs import run_summary, stream_updates, timed


class Checkout(TypedDict, total=False):
    quantity: int
    unit_price: float
    total: float
    status: str
    run_log: Annotated[list[dict], operator.add]


@timed()
def compute(state: Checkout) -> dict:
    return {"total": state["quantity"] * state["unit_price"]}


@timed()
def manual_review(state: Checkout) -> dict:
    return {"status": "held for review"}


@timed()
def approve(state: Checkout) -> dict:
    return {"status": "approved"}


def needs_review(state: Checkout) -> str:
    return "big" if state["total"] > 1000 else "small"


kb = StateGraph(Checkout)
for node in (compute, manual_review, approve):
    kb.add_node(node.__name__, node)
kb.add_edge(START, "compute")
kb.add_conditional_edges("compute", needs_review, {"big": "manual_review", "small": "approve"})
kb.add_edge(["manual_review"], END)
kb.add_edge("approve", END)
checkout = kb.compile()

for qty in (2, 20):
    print(f"quantity={qty}:")
    final = stream_updates(checkout, {"quantity": qty, "unit_price": 89.0})
    print("   status:", final["status"], "| nodes:", list(run_summary(final["run_log"])))
print(checkout.get_graph().draw_mermaid().count(".->"), "conditional edges in the drawing")

**You should now see** for quantity 2: `▸ compute updated: total`, `▸ approve updated: status`, `status: approved`; for 20: `manual_review` instead, `status: held for review`; and `2 conditional edges in the drawing`.

`stream_updates` (from `mastery.graphs`) uses `graph.stream(..., stream_mode=["updates", "values"])`: **updates** tell you which node changed which keys (the best debugging view), **values** give the full state after each step. `@timed()` adds one `{"node", "seconds"}` row per node to `run_log`: the Project 07 cost/latency table starts here.

### What just happened?

`needs_review` returned a label; the dict mapped it to a node. Because the targets were declared, `draw_mermaid()` could draw both dotted edges before anything ran. Without the mapping the graph still runs, but the drawing can't show where the edge may go.

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `InvalidUpdateError: At key 'x': Can receive only one value per step` | Parallel nodes write a key without a reducer | `Annotated[list, operator.add]` (or your own reducer) |
| A list doubles every time you re-run on the previous output | The input passes through the reducer too | Start from a fresh input, or `Overwrite(...)` |
| ``ValueError: Found edge ending at unknown node `zz` `` | Typo in an edge | Node names are function names unless you pass one. (A node with **no** incoming edge compiles fine and simply never runs: check the drawing) |
| Drawing shows no edges out of a router | Targets not declared | `add_conditional_edges(src, fn, ["a", "b"])` or a dict |
| `draw_mermaid_png` hangs or fails | mermaid.ink is unreachable (proxy, offline) | `show_graph` falls back; or `print(graph.get_graph().draw_mermaid())` |
| `ImportError: Install grandalf` | `print_ascii()` needs the extra package | `%pip install grandalf` |

## ⚠️ Production notes

> ⚠️ **Production note: state is a contract.**
> Once runs are checkpointed (Module 08), old state must still load after you deploy new code. Add keys with defaults; don't rename or change types silently.

> ⚠️ **Production note: validate at the boundary.**
> Use a Pydantic input schema (or validate in your API) for data that comes from users; keep internal state lean with `TypedDict`.

## 🏋️ Exercises

1. **(Easy)** Add a `tax` node between `price` and `confirm` in `order_graph` (19% VAT). Redraw the graph.
2. **(Medium)** Write a reducer `max_value(old, new)` that keeps the larger number, and use it for a `best_score` key updated by two parallel nodes.
3. **(Medium)** In `ChatState`, what happens if a node returns `{"messages": [AIMessage("x", id="m1")]}` where `m1` is the first user message? Try it.
4. **(Stretch)** Change `checkout` so the threshold (1000) comes from `context` instead of being hard-coded.

<details>
<summary><b>Show solutions</b></summary>

**1.** `def tax(state): return {"total": round(state["total"] * 1.19, 2)}`; edges `price → tax → confirm`. `show_graph(order_graph)` shows three nodes.

**2.** `def max_value(old, new): return new if old is None else max(old, new)`; `best_score: Annotated[int, max_value]`. Two nodes from `START` writing 3 and 7 give `best_score == 7`.

**3.** `add_messages` **replaces** the message with id `m1`: the user's message becomes an AI message. Ids matter; let LangChain generate them unless you mean to replace.

**4.** `@dataclass class Limits: review_above: float = 1000`; `StateGraph(Checkout, context_schema=Limits)`; `def needs_review(state, runtime: Runtime[Limits])`; `invoke(..., context=Limits(500))`.
</details>

## ✅ Checkpoint

- [ ] I can build, compile, invoke and draw a `StateGraph`.
- [ ] I can explain overwrite vs reducer, and fix `InvalidUpdateError`.
- [ ] I used `add_messages` / `MessagesState` and runtime context.
- [ ] I read a run with `stream_mode="updates"`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

The [Graph API overview](https://docs.langchain.com/oss/python/langgraph/graph-api) describes **input/output schemas** and **private state** (keys only some nodes see). When would you use an output schema in this course's projects? Log your answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- State schema + nodes (partial updates) + edges = a graph. `invoke` returns the final state.
- Reducers decide merges: overwrite by default; `operator.add`, `add_messages`, or your own function via `Annotated`.
- Conditional edges route on state; declare their targets so the graph can be drawn and checked.
- `show_graph`, `draw_mermaid()` and `stream_mode="updates"` are your debugging tools.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.03 · Control flow: `Command`, `Send`, cycles, recursion limits** → [`07-03-control-flow.ipynb`](07-03-control-flow.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)